# プロンプトレスキューワークショップ

## 開発から3週間。精度は19%。レビューは明日。

お客様の**TechSupport Corp**は、Claudeを使ったサポートチケット処理システムを構築しました。P1〜P4の優先度分類、エンティティ抽出、回答の下書き作成を、1つのプロンプトにまとめています。整ったチケットではうまく動きますが、実際の入力では破綻します。優先度の誤り、ハルシネーションされたエンティティ、壊れたJSON、筋の通らない回答が出ます。

明日のレビューの前に、そして誰かがモデルの入れ替えを提案する前に、何が本当に壊れているのかを突き止める必要があります。それが今回のセッションです。

**あなたの仕事：**失敗を診断し、問題を修正し、判断の理由をはっきり説明できるようにすることです。説明できないスコアの向上は、デモにすぎません。診断の裏付けがあって初めて、方法論になります。

### 学習目標

このワークショップを終えると、次のことができるようになります。

1. プロンプトが失敗する根本原因を**診断**する。症状だけでなく、*構造的な*理由まで突き止めます
2. それぞれの選択に明確な根拠を添えて、プロンプトエンジニアリングの手法を**適用**する
3. 定量的な評価ハーネスで改善を**評価**する。感覚に頼った判断はしません
4. 評価結果という証拠をもとに、設計上の判断を**擁護**する
5. 自分が設計した部分とClaudeが担った部分を**振り返る**。お客様との会話で重要になる区別です

### 進め方

これは**個人演習**です。以下のセルを自分のペースで進めてください。

**おすすめの流れ：**まず短く診断し、時間の大半を構築に使い、最後に短く振り返ります。*（セッションの時間配分はファシリテーターが決めます。）*

---

**Partner Basecamp参加者向けに作成しています。** 研修資料としての複製・再配布はできません。ここで学ぶパターンを、ご自身のお客様向けの業務に活用することは自由です。

In [ ]:
# このカーネルに依存パッケージをインストールします。再実行しても安全で、制限のかかったPython（PEP 668）でも動作します。
import importlib.util, os, subprocess, sys

# ── 環境ガード（Environment guard）─────────────────────────────────
# 下のPythonのインストール処理と同じセルに置いてあるので、飛ばせません。
# システムのPythonそのものにパッケージがインストールされることはありません（以前の最悪のケースは、
# IT部門が管理するマシンに対する--break-system-packagesでした）。ここで止まった場合は、
# SETUP.mdの "Why the notebook just stopped" を参照してください。
def _in_isolated_env():
    """実行中のカーネルが venv／virtualenv／conda環境、またはColabの場合にTrueを返します。
    判定はインタープリタ自身（sys.*）から行います。起動元のシェルから継承した
    有効化用の環境変数は、sys.executableが実際にその変数の指す環境の内側にある
    場合に限って信頼します。有効化済みのターミナルから起動したシステムPythonの
    カーネルでも VIRTUAL_ENV は継承されますが、それを通してはいけません。"""
    if "google.colab" in sys.modules:
        return True  # Colabは使い捨ての独立したランタイムを自前で用意する
    if sys.prefix != getattr(sys, "base_prefix", sys.prefix):
        return True  # PEP 405のvenv（python -m venv）。condaの環境の大半も該当する
    if hasattr(sys, "real_prefix"):
        return True  # 旧式のvirtualenv
    exe = os.path.realpath(sys.executable)
    for var in ("VIRTUAL_ENV", "CONDA_PREFIX"):
        root = os.environ.get(var)
        if not root or not exe.startswith(os.path.realpath(root) + os.sep):
            continue  # シェル（shell）から受け取っただけの伝聞で、カーネルは別の場所で動いている
        if var == "CONDA_PREFIX" and os.environ.get("CONDA_DEFAULT_ENV", "base") == "base":
            continue  # condaの共有`base`は隔離環境とはみなさない
        return True
    return False

if os.environ.get("BASECAMP_ALLOW_SYSTEM_PYTHON") == "1":
    print("⚠️  環境ガードをバイパスしました（BASECAMP_ALLOW_SYSTEM_PYTHON=1）。"
          "意図的にこのPythonへインストールします。")
elif not _in_isolated_env():
    _HEAD = "✗ システムPythonを検出しました。何もインストールせずに停止しました"
    _BODY = (
        "ここにパッケージをインストールすると、マシン全体のPythonが変わります。会社管理のノートPCでは、\n"
        "IT部門への問い合わせが必要になることがあります。\n"
        "\n"
        "対処方法（2ステップ）：\n"
        "  1. ターミナルで、リポジトリのルートから次を実行します：\n"
        "       python3 -m venv .venv\n"
        "       source .venv/bin/activate          # Windowsの場合: .venv\\Scripts\\activate\n"
        "       pip install -r requirements.txt\n"
        "  2. VS Codeで、右上のカーネル名をクリックし、Select Another Kernel →\n"
        "     Python Environments → 末尾が.venvのものを選び → このセルをもう一度実行します。\n"
        "\n"
        "condaを使っていますか？ `conda activate <env>`（`base`ではありません）を実行してから、そのカーネルを選びます。\n"
        "自己管理のマシンを使うファシリテーターの場合：BASECAMP_ALLOW_SYSTEM_PYTHON=1 でバイパスできます。"
    )
    _shown = False
    try:  # APIキー確認（下）と同じバナー表現（緑のボックス、赤はエラー用）
        from IPython import get_ipython
        if get_ipython().__class__.__name__ == "ZMQInteractiveShell":
            import html as _html
            from IPython.display import HTML, display
            display(HTML(
                '<div style="padding:12px 16px;border-radius:8px;background:#fdecea;'
                'border:1.5px solid #b42318;font-size:15px;font-family:sans-serif;">'
                '<div style="color:#b42318;font-weight:600;">' + _html.escape(_HEAD) + '</div>'
                '<pre style="margin:10px 0 0;font-family:inherit;font-size:14px;font-weight:400;'
                'color:#141413;white-space:pre-wrap;">' + _html.escape(_BODY) + '</pre></div>'
            ))
            _shown = True
    except Exception:
        pass
    raise SystemExit(
        "環境ガードがこのセルを停止しました。上のメッセージを確認してください。"
        if _shown else "\n  " + _HEAD + "\n\n" + _BODY + "\n"
    )
else:
    print("✓ 環境ガード：隔離されたインタープリタを検出しました。インストールしても安全です")
# ──────────────────────────────────────────────────────────────────────────


def _ensure_packages(requirements):
    """requirements：(import_name, pip_spec) のリスト。足りないものだけを、
    実行中のインタープリタにインストールします。通常のインストール、ユーザー領域、
    PEP 668の上書きの順に試します（ユーザー領域が先で、システム全体は最後の手段）。
    どの試行もサイレントで、pipの出力はストリーム表示せず取得するだけです。そのため、
    制限のかかったPython（HomebrewやDebian、PEP 668）でも、フォールバックで実際には
    成功しているのに 'externally-managed-environment' の恐ろしげな長文が出ることは
    もうありません。すべての方法が失敗したときだけ、生のトレースバックではなく、
    venvでの対処方法とともに理由を表示します。"""
    missing = [pip for mod, pip in requirements if importlib.util.find_spec(mod) is None]
    if not missing:
        return
    print("Installing " + ", ".join(missing) + "（初回のみです。しばらくお待ちください）…", flush=True)
    base = [sys.executable, "-m", "pip", "install", "-q"]
    last = None
    for extra in ([], ["--user"], ["--user", "--break-system-packages"], ["--break-system-packages"]):
        last = subprocess.run(base + extra + missing, capture_output=True, text=True)
        if last.returncode == 0:
            return
    pip_said = (last.stderr or last.stdout or "").strip().splitlines() if last else []
    tail = "\n      ".join(pip_said[-3:]) if pip_said else "（pipからの出力なし）"
    raise SystemExit(
        "\nインストールできませんでした：" + ", ".join(missing) + "\n"
        "  このPythonは制限されている（PEP 668）か、オフラインです。最も簡単な対処はvenvを使うことです：\n"
        f"      {sys.executable} -m venv .venv\n"
        "      source .venv/bin/activate          # Windowsの場合: SETUP.mdを参照\n"
        f"      pip install {' '.join(missing)}\n"
        "  その後、VS Code（右上のカーネル選択）で.venvのインタープリタを選び、Run Allを実行します。\n"
        "  社内プロキシがある、またはPyPIがブロックされている場合は、リポジトリのルートにあるSETUP.mdを参照してください。\n"
        f"  (pip said: {tail})\n"
    )

_ensure_packages([("anthropic", "anthropic"), ("matplotlib", "matplotlib")])
print("✓ 依存パッケージの準備ができました")

### セットアップ：Claudeに接続する

何かをレスキューする前に、まず接続を確立する必要があります。次のセルを最初に実行してください。初回実行時に、セットアップのセルが**`.env`ファイル**を作成します（gitignore済みなので、キーがコミットされることはありません）。そのファイルを開き、`ANTHROPIC_API_KEY=`の後ろにキーを貼り付けて保存し、もう一度実行してください。カーネルを再起動しても残るので、貼り付けは1回で済みます。*（`.env`がまだない場合は、代わりに非表示の入力ボックスが表示されます。）* 緑色の**「✓ APIキーを確認しました」**バナーが表示されれば準備完了です。赤いバナーが出た場合は、表示された指示に従って、もう一度セルを実行してください。

In [ ]:
import os

def _status(ok, msg):
    """ノートブックでは緑／赤のバナー、スクリプトとして実行した場合はプレーンテキストで表示します。"""
    try:
        from IPython import get_ipython
        shell = get_ipython()
        if shell is None or shell.__class__.__name__ != "ZMQInteractiveShell":
            raise RuntimeError("ノートブックのカーネルではありません。プレーンテキストのバナーを使います")
        from IPython.display import display, HTML
        color = "#1a7f37" if ok else "#b42318"
        bg = "#e6f4ea" if ok else "#fdecea"
        icon = "✓" if ok else "✗"
        display(HTML(
            f'<div style="padding:12px 16px;border-radius:8px;background:{bg};'
            f'border:1.5px solid {color};color:{color};font-weight:600;'
            f'font-size:15px;font-family:sans-serif;">{icon} {msg}</div>'
        ))
    except Exception:
        print(("[OK] " if ok else "[!!] ") + msg)

import os
import pathlib

import anthropic

# ── Claudeに接続する：Anthropic APIまたはAmazon Bedrock ──
# どちらの認証情報（API key／token）でも使えます。どちらを持っているかは、このセルが判別します。
#   Anthropic API : ANTHROPIC_API_KEY=sk-ant-...
#   Amazon Bedrock: AWS_BEARER_TOKEN_BEDROCK=...  plus  AWS_REGION=us-east-1
# 使う方を、このセルが作成する.envファイル（gitignore済みなので、コミットされることはありません）に書くか、
# シェルでexportしてください。シェルの値は、.envファイルの値より優先されます。
_ENV_TEMPLATE = (
    "# Anthropic API key — paste after the = (no quotes, no spaces), then save and\n"
    "# re-run the setup cell. Get one at https://console.anthropic.com/\n"
    "ANTHROPIC_API_KEY=paste-your-key-here\n"
    "\n"
    "# --- Using Amazon Bedrock instead? Comment out the line above and fill these in:\n"
    "# AWS_BEARER_TOKEN_BEDROCK=paste-your-bedrock-api-key-here\n"
    "# AWS_REGION=us-east-1\n"
)


def _resolve_env_file():
    """.envを、作業ディレクトリから上へたどって、最も近い既存のものを返します（ルートの.env 1つで
    すべての演習をまかなえます）。まだ無ければリポジトリのルート、ノートブックだけを開いている
    場合はこのフォルダを指します。"""
    here = pathlib.Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / ".env").is_file():
            return d / ".env"
    root = next((d for d in [here, *here.parents]
                 if (d / "SETUP.md").exists() or (d / ".git").exists()), here)
    return root / ".env"


_env_file = _resolve_env_file()
if not _env_file.exists():
    _env_file.write_text(_ENV_TEMPLATE)
    print(f"Created {_env_file.name} in {_env_file.parent} — open it, add your key, "
          "save, then re-run this cell.")

# 小さな.envパーサーです（python-dotenvは不要）。実行のたびに読み直すので、キーを貼り付けて
# 再実行すれば反映されます。環境（シェル／Claude Code／CI）にある本物の値が優先され、
# プレースホルダー（paste-your-key-here）が居座ることはありません。
_file = {}
for _line in (_env_file.read_text().splitlines() if _env_file.exists() else []):
    _line = _line.strip()
    if _line and not _line.startswith("#") and "=" in _line:
        _k, _v = _line.split("=", 1)
        _file[_k.strip()] = _v.strip().strip('"').strip("'")
for _k, _v in _file.items():
    if _k != "ANTHROPIC_API_KEY":
        os.environ.setdefault(_k, _v)

_shell_key = os.environ.get("ANTHROPIC_API_KEY", "").strip()
_anthropic_key = _shell_key if _shell_key.startswith("sk-ant-") else _file.get("ANTHROPIC_API_KEY", "").strip()
_bedrock_token = os.environ.get("AWS_BEARER_TOKEN_BEDROCK", "").strip()
_bedrock_region = os.environ.get("AWS_REGION", "").strip()

if _anthropic_key.startswith("sk-ant-"):
    PROVIDER = "anthropic"
elif _bedrock_token:
    PROVIDER = "bedrock"
else:
    PROVIDER = None


def _needs_credentials(head, body):
    """警告色（黄）のバナーを出して停止します。セットアップの失敗を、数セル先ではなくここで起こすためです。"""
    _shown = False
    try:
        from IPython import get_ipython
        if get_ipython().__class__.__name__ == "ZMQInteractiveShell":
            import html as _html
            from IPython.display import HTML, display
            display(HTML(
                '<div style="padding:12px 16px;border-radius:8px;background:#fff8c5;'
                'border:1.5px solid #9a6700;font-size:15px;font-family:sans-serif;">'
                '<div style="color:#9a6700;font-weight:600;">' + _html.escape(head) + '</div>'
                '<pre style="margin:10px 0 0;font-family:inherit;font-size:14px;font-weight:400;'
                'color:#141413;white-space:pre-wrap;">' + _html.escape(body) + '</pre></div>'
            ))
            _shown = True
    except Exception:
        pass
    if not _shown:
        print("\n" + head + ":\n   " + body.replace("\n", "\n   ") + "\n")
    raise SystemExit("認証情報がありません。上のメッセージを確認してください。")


if PROVIDER is None:
    _needs_credentials(
        "📋 続行するには認証情報を追加してください",
        f"Open this file:  {_env_file}\n"
        "\n"
        "Anthropic APIを使う場合は、次を設定します：\n"
        "    ANTHROPIC_API_KEY=sk-ant-...\n"
        "\n"
        "Amazon Bedrockを使う場合は、次の両方を設定します：\n"
        "    AWS_BEARER_TOKEN_BEDROCK=<your Bedrock API key>\n"
        "    AWS_REGION=us-east-1          # モデルを有効にしているリージョン\n"
        "\n"
        "ファイルを保存してから、このセルの▶をもう一度クリックしてください。"
    )

if PROVIDER == "bedrock" and not _bedrock_region:
    _needs_credentials(
        "📋 Bedrockにはリージョンが必要です",
        f"Found AWS_BEARER_TOKEN_BEDROCK but no AWS_REGION.\n"
        f"\n"
        f"Open this file:  {_env_file}\n"
        "Bedrockのモデルを有効にしているリージョンを追加してください。例：\n"
        "    AWS_REGION=us-east-1\n"
        "\n"
        "ファイルを保存してから、このセルの▶をもう一度クリックしてください。"
    )


def _model(name):
    """BedrockのモデルIDには`anthropic.`というプレフィックスが付きます。Anthropic APIでは、付けないIDを使います。"""
    return f"anthropic.{name}" if PROVIDER == "bedrock" else name


# この演習で使うClaudeのモデルに名前を付けます。使っているプロバイダーに合わせて解決されます。
MODEL = _model("claude-sonnet-5")        # この演習の主力（workhorse）モデル
FAST_MODEL = _model("claude-haiku-4-5")  # 安価で高速（接続確認、judge用）
BIG_MODEL = _model("claude-opus-4-8")    # より大きなモデル（Opus）を試したいときに使う


def _make_client(timeout, max_retries=2):
    if PROVIDER == "bedrock":
        from anthropic import AnthropicBedrockMantle
        return AnthropicBedrockMantle(aws_region=_bedrock_region,
                                      timeout=timeout, max_retries=max_retries)
    return anthropic.Anthropic(api_key=_anthropic_key,
                               timeout=timeout, max_retries=max_retries)


# 接続確認（connection check）：認証情報と、このモデルにあなたのアカウントから到達できるかの両方を検証します。
# Bedrockでは、キーが有効でも、モデルがアカウントやリージョンで有効化されていないと404になることがあります。
# そのため、認証情報の形式だけを見るのではなく、実際のモデルIDにpingを送ります。
_probe = _make_client(timeout=30.0, max_retries=1)
try:
    _probe.messages.create(model=FAST_MODEL, max_tokens=1,
                           messages=[{"role": "user", "content": "ping"}])
except anthropic.NotFoundError:
    if PROVIDER == "bedrock":
        _status(False, f"Bedrock reached, but model '{FAST_MODEL}' isn't available to you in "
                       f"{_bedrock_region}. Enable model access for it in the Bedrock console "
                       f"(or switch AWS_REGION to a region where it is enabled), then re-run.")
    else:
        _status(False, f"Model '{FAST_MODEL}' not found for this key.")
    raise SystemExit("モデルを利用できません。上のメッセージを確認してください。")
except (anthropic.AuthenticationError, anthropic.PermissionDeniedError):
    if PROVIDER == "bedrock":
        _status(False, "そのBedrockキーは拒否されました。AWS_BEARER_TOKEN_BEDROCKと、"
                       "Bedrockの呼び出し権限があることを確認してから、このセルをもう一度実行してください。")
    else:
        _status(False, "そのキーは拒否されました。このセルをもう一度実行し、キー全体を"
                       "貼り付けてください（sk-ant-で始まります）。")
    raise SystemExit("認証情報が受け付けられませんでした。このセルをもう一度実行してやり直してください。")
except Exception as exc:
    _status(False, "APIに接続できませんでした（" + type(exc).__name__ + "）。"
                   "接続を確認してから、このセルをもう一度実行してください。")
    raise
else:
    if PROVIDER == "anthropic":
        os.environ["ANTHROPIC_API_KEY"] = _anthropic_key  # 以降のセルや!pythonがこれを参照する
        _status(True, "APIキーを確認しました。Claudeに接続されています。")
    else:
        _status(True, f"Bedrock key verified ({_bedrock_region}) - you're connected to Claude "
                      f"as {MODEL}.")

# 実際に使うクライアント。タイムアウトは長めにしています。非ストリーミング呼び出しでmax_tokensが21333を超える場合に必要です。
client = _make_client(timeout=900.0)


## シナリオ：TechSupport Corp

### お客様からの依頼

> 「私たちはエンタープライズSaaS企業のTechSupport Corpです。届いたサポートチケットの優先度を分類し、重要な情報を抽出し、回答を下書きするプロンプトを、1回の呼び出しで処理する形で構築しました。単純で分かりやすく書かれたチケットでは、とてもうまく動きます。ところが本番では、チケットの半分ほどが乱雑です。複数の問題が混ざっていたり、説明が曖昧だったり、英語が母語でない方からのものだったりします。そうしたチケットで、ひどい失敗が起きています。JSONが壊れて返ってきて、優先度は間違い、下書きの回答が分類結果と矛盾することもあります。更新の打ち合わせまでに、これを直す必要があります。制約が1つあります。チケット処理のパイプラインには、1チケットあたり5秒のレイテンシ上限があります。そのため、逐次のAPI呼び出しが3回を超える解決策には、その正当性の説明が必要です。」

### チャレンジ

1. 失敗の根本原因を**診断**する（症状だけでなく、このプロンプトの構造が*なぜ*こうした失敗を生むのかを示す）
2. 手持ちの道具（下の「ツールキット」を参照）から、少なくとも2つの手法を使ってプロンプトを**修正**する
3. 評価スイートで修正を**検証**する
4. 3分間の「お客様向け報告」を**発表**する。診断、修正、結果、再発防止のアドバイスを含めます

### 成功基準

| レベル | 目標 |
|-------|--------|
| **ベースライン（合格）** | 評価の合格率75%以上、根本原因の特定、根拠を添えた2つ以上の手法の使用、筋の通った報告 |
| **Tier 1：オプティマイザー** | 合格率90%以上、記録された反復ログ、モニタリング用プロンプトの設計 |
| **Tier 2：アーキテクト** | 自己修復するプロンプトチェーン（品質チェックに失敗したら自動で再試行） |

### 制約

- **モデル：** Claude Haiku（評価ハーネスにハードコード済み。対策としてモデルのアップグレードは不可）
- **アーキテクチャ：** プロンプトは全面的に作り直して構いません。複数ステップやチェーンへの分割も可能です。ただし、エンドツーエンドの機能は同じままにします。同じ入力から、同じ種類の出力が得られる必要があります
- **プロセス：** プロンプトの反復を1回ごとに記録してください。評価ハーネスが、あなたの進捗を自動で記録します

### ✏️ あなたの番：これが、あなたが直すプロンプトです

**ここ**で編集して、評価を再実行し、スコアが動くのを確認してください。

In [ ]:
# ✏️ あなたの番（YOUR TURN）：このプロンプトを編集してください
# ============================================================
# あなたのプロンプト（YOUR PROMPT）：壊れたプロンプトを直すために、ここを編集してください！
# ============================================================

system_prompt = """
あなたはサポートチケットの処理担当です。各チケットについて、次のことを行ってください。
1. ビジネスへの影響に基づいて優先度（P1-P4）を分類する
2. 抽出する：製品名、バージョン、エラーコード、影響を受けるユーザー数
3. 問題を認め、次のステップを示す、役に立つ回答の下書きを作成する
4. すべてをJSONで返す: {"priority": "", "entities": {"product": "", "version": "", "error_codes": [], "affected_users": ""}, "response": "", "confidence": "high/medium/low"}

ルール:
- P1 = システム停止、全ユーザーに影響
- P2 = 主要機能の障害、多くのユーザーに影響
- P3 = 軽微なバグ、影響を受けるユーザーは少数
- P4 = 機能リクエストまたは見た目の問題
- 優先度に迷う場合は、あなたの最善の判断で決める
- 回答はプロフェッショナルで、共感のこもったものにする
- 空であっても、すべてのJSONフィールドを必ず含める
- 簡潔に、ただし十分に書く
"""

print("壊れたプロンプトを読み込みました。下の評価スイートのセルを実行して、ベースラインのスコアを確認してください。")
print(f"Prompt length: {len(system_prompt)} characters")

## 失敗の例：診断のための材料

以下は、壊れたプロンプトが失敗する実際の入力5件です。修正に取りかかる前に、パターンをよく調べてください。

---

### 例1：複数の問題を含むチケット

**入力：**
```
Subject: URGENT - billing broken AND can't export data

Our billing dashboard has been showing wrong numbers since Tuesday and also the CSV export
feature throws a 500 error. We have board reporting on Friday and need both fixed. About 200
users on our team are affected by the export issue, billing is just our finance team (3 people)
but it's blocking payroll.
```

**何が問題だったか：**P2に分類されます（給与処理が止まっているのでP1であるべきです）。2つの問題を1つのエンティティ抽出にまとめてしまいます。回答もエクスポートの問題にしか触れず、請求と給与処理の問題を無視しています。

---

### 例2：曖昧なチケット

**入力：**
```
Subject: things arent working right

hey so a bunch of stuff seems off today? like the pages load slow and sometimes i get errors.
not sure whats happening. its been like this since the morning. thanks
```

**何が問題だったか：**入力のどこにも存在しない製品名「PageLoader」とエラーコード「ERR_SLOW_LOAD」をハルシネーションします。自信を持って分類できるだけの情報がないのに、P3に分類してしまいます。

---

### 例3：英語が母語でない人からのチケット

**入力：**
```
Subject: Probleme mit der Integration

Hello, we have problem with API integration since update to version 3.2. The webhook is not
fire correctly and data is missing in dashboard. Very urgent for us because is production
system for 50 customer. Error say "timeout exceeded" sometimes.
```

**何が問題だったか：**JSONの出力が壊れています。responseフィールドに含まれる引用符がJSON構造を壊します。優先度は正しい（P2）ものの、エンティティでバージョン番号が抜けています。

---

### 例4：バグを装った機能リクエスト

**入力：**
```
Subject: BUG: No dark mode support

This is a critical usability bug. Your application doesn't support dark mode which causes eye
strain for our entire team working night shifts (15 people). This should have been implemented
long ago. We need this resolved immediately or we'll consider switching providers.
```

**何が問題だったか：**緊急を訴える言葉につられてP2に分類されます。これは明らかにP4（機能リクエスト）です。回答では「バグを修正します」と約束してしまい、お客様に誤った期待を与えます。

---

### 例5：整ったチケット（回帰チェック）

**入力：**
```
Subject: Error 503 on login page

Since 2pm EST today, all users (approximately 500) in our organization are unable to log into
the platform. We receive a 503 Service Unavailable error. This is our primary CRM tool and
sales team is completely blocked. Error code: SVC-503-AUTH. Running version 4.1.2.
```

**何が問題だったか：**実際には、入力が整っているため、ほとんどの場合は正しく処理できます。簡単なケースで修正が後退していないかを確認するために含めています。

In [ ]:
#@title 評価ハーネス（クリックして読み込み。このセルは編集しないでください）

# ============================================================
# 評価ハーネス（EVAL HARNESS）：すべての評価ロジックがここに埋め込まれています。
# このセルを1回実行して、ハーネス（harness）の関数を読み込みます。
# ============================================================

import json
import re
import base64
import time
from datetime import datetime
import anthropic
def _in_notebook():
    """Jupyterカーネル（VS Code／JupyterLab）の中でのみTrueを返します。ここではリッチなHTMLやチャートが表示されます。"""
    try:
        from IPython import get_ipython
        shell = get_ipython()
        return shell is not None and shell.__class__.__name__ == "ZMQInteractiveShell"
    except Exception:
        return False

def _viz():
    """チャートはノートブック向けの装飾なので、描画するときだけmatplotlibを遅延読み込みします。"""
    try:
        import matplotlib
        import matplotlib.pyplot as plt
    except ImportError:
        print("[チャートをスキップ] matplotlibがインストールされていません。表示するには python3 -m pip install matplotlib を実行してください")
        return None
    matplotlib.rcParams['figure.facecolor'] = 'white'
    return plt

def _show_html(html):
    """ノートブックではリッチなスコアカードを表示し、ターミナルでは同じ数値をプレーンテキストで表示します。"""
    if _in_notebook():
        from IPython.display import display, HTML
        display(HTML(html))

# ---------------------------------------------------------------------------
# 埋め込みの評価ケース（うっかり覗かれないよう、base64でエンコードしています）
# ---------------------------------------------------------------------------

_EVAL_CASES_B64 = "..."

# ---------------------------------------------------------------------------
# グローバル状態（global state）
# ---------------------------------------------------------------------------

_iteration_log = []
_baseline_score = None
_baseline_total = None

MODEL = FAST_MODEL  # この演習はHaikuで実行する。プロバイダーに合わせたIDはセットアップで解決済み
MAX_TOKENS = 4096

# ---------------------------------------------------------------------------
# ケースローダー（case loader）
# ---------------------------------------------------------------------------

def _load_cases(encoded):
    return json.loads(base64.b64decode(encoded).decode("utf-8"))

# ---------------------------------------------------------------------------
# スコアリングモジュール（scoring.pyから移植）
# ---------------------------------------------------------------------------

REQUIRED_FIELDS = {"priority", "entities", "response", "confidence"}
REQUIRED_ENTITY_FIELDS = {"product", "version", "error_codes", "affected_users"}


def parse_output(raw_output):
    """モデルの出力からJSONを抽出します。ネストしたオブジェクトも扱います。"""
    try:
        parsed = json.loads(raw_output.strip())
        return parsed, None
    except json.JSONDecodeError:
        pass
    start_indices = [i for i, c in enumerate(raw_output) if c == '{']
    for start in start_indices:
        brace_count = 0
        for end in range(start, len(raw_output)):
            if raw_output[end] == '{':
                brace_count += 1
            elif raw_output[end] == '}':
                brace_count -= 1
            if brace_count == 0:
                candidate = raw_output[start:end + 1]
                try:
                    parsed = json.loads(candidate)
                    return parsed, None
                except json.JSONDecodeError:
                    break
    return None, "出力が有効なJSONではありません"


def check_valid_json(parsed, error):
    if parsed is None:
        return False, error or "出力が有効なJSONではありません"
    missing_top = REQUIRED_FIELDS - set(parsed.keys())
    if missing_top:
        return False, f"Missing required fields: {', '.join(sorted(missing_top))}"
    entities = parsed.get("entities")
    if not isinstance(entities, dict):
        return False, "フィールド'entities'はJSONオブジェクトである必要があります"
    missing_entity = REQUIRED_ENTITY_FIELDS - set(entities.keys())
    if missing_entity:
        return False, f"Missing entity fields: {', '.join(sorted(missing_entity))}"
    return True, "有効なJSONで、必須フィールドがすべてそろっています"


def check_priority(parsed, gold_priority):
    if parsed is None:
        return False, "優先度を確認できません：JSONが無効です"
    output_priority = str(parsed.get("priority", "")).strip().upper()
    expected = gold_priority.strip().upper()
    if output_priority == expected:
        return True, f"Priority correct: {expected}"
    return False, f"Priority={output_priority}, expected={expected}"


def _normalize_value(val):
    return re.sub(r'[\s\-_\.]+', ' ', str(val).lower().strip())


_NULL_EQUIVALENT_PATTERN = re.compile(
    r'^('
    r'null|none|n/?a|unknown|unspecified|not\s+specified'
    r'|not\s+provided|not\s+mentioned|not\s+available'
    r'|unknown\s*[-\u2014\u2013]\s*not\s+\w+'
    r'|\w+\s*\(unspecified[^)]*\)'
    r')$',
    re.IGNORECASE,
)


def _is_null_equivalent(val_str):
    """値が意味的にnullである場合（モデルが「存在しない」と言っている場合）にTrueを返します。"""
    return bool(_NULL_EQUIVALENT_PATTERN.match(val_str.strip()))


def _strip_annotations(val):
    """括弧付きの接尾辞と、先頭の近似マーカーを取り除きます。"""
    val = re.sub(r'\s*\(.*?\)\s*$', '', val)
    val = re.sub(r'^[~\u2248]\s*', '', val)
    return val.strip()


def _value_in_input(value, input_text):
    """エンティティの値が入力テキストから導き出せるかを確認します。"""
    if value is None or value == "" or value == []:
        return True
    input_lower = input_text.lower()
    input_normalized = _normalize_value(input_text)
    if isinstance(value, list):
        return all(_value_in_input(v, input_text) for v in value)
    val_str = str(value).strip()
    if not val_str or _is_null_equivalent(val_str):
        return True
    val_lower = val_str.lower()
    val_normalized = _normalize_value(val_str)
    if val_lower in input_lower:
        return True
    if val_normalized in input_normalized:
        return True
    if re.match(r'^\d+$', val_str):
        if re.search(r'\b' + re.escape(val_str) + r'\b', input_text):
            return True
    if re.match(r'^\d+\.\d+(\.\d+)?$', val_str):
        if val_str in input_text:
            return True
    # フォールバック：注釈（annotation）を取り除いて再試行する
    stripped = _strip_annotations(val_str)
    if stripped and stripped != val_str:
        if _value_in_input(stripped, input_text):
            return True
    # フォールバック：入力中に近似表現（approximately など）がある数値
    num_match = re.match(r'^[~\u2248]?\s*(\d+(?:\.\d+)?)\s*$', val_str)
    if num_match:
        number = num_match.group(1)
        approx_pattern = (
            r'(?:approximately|about|around|roughly|~|\u2248)\s*'
            + re.escape(number)
        )
        if re.search(approx_pattern, input_text, re.IGNORECASE):
            return True
        if re.search(r'\b' + re.escape(number) + r'\b', input_text):
            return True
    # フォールバック：カンマ区切り（comma-separated）の複数の値
    if ',' in val_str:
        parts = [p.strip() for p in val_str.split(',') if p.strip()]
        if len(parts) > 1 and all(
            _value_in_input(part, input_text) for part in parts
        ):
            return True
    return False


def check_entities(parsed, input_text, gold_entities):
    if parsed is None:
        return False, "エンティティを確認できません：JSONが無効です"
    entities = parsed.get("entities", {})
    if not isinstance(entities, dict):
        return False, "entitiesフィールドはJSONオブジェクトではありません"
    hallucinated = []
    for field in REQUIRED_ENTITY_FIELDS:
        value = entities.get(field)
        if value is None or value == "" or value == []:
            continue
        if not _value_in_input(value, input_text):
            hallucinated.append(f"{field}={json.dumps(value)}")
    if hallucinated:
        return False, f"Possible hallucinated entities: {'; '.join(hallucinated)}"
    return True, "すべてのエンティティが入力から導き出せます"


def score_case(parsed, parse_error, input_text, gold_priority, gold_entities,
               response_coherent=None, audited=False):
    json_ok, json_reason = check_valid_json(parsed, parse_error)
    priority_ok, priority_reason = (False, "スキップ：JSONが無効です") if not json_ok else check_priority(parsed, gold_priority)
    entities_ok, entities_reason = (False, "スキップ：JSONが無効です") if not json_ok else check_entities(parsed, input_text, gold_entities)
    if not audited:
        response_ok = True
        response_reason = "自動合格（監査対象外のケース）"
    elif not json_ok:
        response_ok = False
        response_reason = "スキップ：JSONが無効です"
    elif response_coherent is None:
        response_ok = True
        response_reason = "ジャッジの評価待ち"
    else:
        response_ok = response_coherent
        response_reason = "ジャッジ：PASS" if response_coherent else "ジャッジ：FAIL"
    all_pass = json_ok and priority_ok and entities_ok and response_ok
    return {
        "pass": all_pass,
        "criteria": {
            "json_valid": {"pass": json_ok, "reason": json_reason},
            "priority_correct": {"pass": priority_ok, "reason": priority_reason},
            "entities_accurate": {"pass": entities_ok, "reason": entities_reason},
            "response_coherent": {"pass": response_ok, "reason": response_reason},
        },
    }

# ---------------------------------------------------------------------------
# ジャッジプロンプトモジュール（judge_prompts.pyから移植）
# ---------------------------------------------------------------------------

JUDGE_SYSTEM_PROMPT = """あなたは、サポートチケット処理システムの評価ジャッジです。あなたの仕事は、優先度の分類を踏まえて、下書きされた回答がサポートチケットに適切に対応しているかを判定することです。

ルール：
- 回答は、優先度の分類と矛盾してはならない
- 回答は、チケットの実際の内容に対応していなければならない
- P4に分類された機能リクエストの場合：回答は、それを「修正」すると約束したり、バグとして扱ったりしてはならない
- 曖昧なチケットの場合：回答は、追加の情報を求めるべきである
- 複数の問題を含むチケットの場合：回答は、挙げられたすべての問題に触れるべきである
- チケットの口調にかかわらず、口調はプロフェッショナルであるべきである

ちょうど1行で返してください。英語の PASS または FAIL で始め、その後に1文の理由を続けます。
例：PASS - 回答は機能リクエストを適切に受け止め、期待値を設定しています。
例：FAIL - 回答は機能リクエストをバグのように扱い、修正を約束しています。"""


def judge_response(client, input_text, priority, response_text):
    user_message = f"""Evaluate this support ticket response:

SUPPORT TICKET:
{input_text}

ASSIGNED PRIORITY: {priority}

DRAFTED RESPONSE:
{response_text}

Does the response appropriately address the ticket without contradicting the classification? Reply PASS or FAIL with a one-sentence reason."""
    try:
        result = _call_api_with_retry(
            client, model=MODEL, max_tokens=150, system=JUDGE_SYSTEM_PROMPT,
            messages=[{"role": "user", "content": user_message}],
        )
        judge_output = result.content[0].text.strip()
        if judge_output.upper().startswith("PASS"):
            return True, judge_output
        elif judge_output.upper().startswith("FAIL"):
            return False, judge_output
        else:
            return True, f"Ambiguous judge output (defaulting to PASS): {judge_output}"
    except Exception as e:
        return True, f"Judge call failed (defaulting to PASS): {e}"

# ---------------------------------------------------------------------------
# 評価の実行本体（core eval runner）
# ---------------------------------------------------------------------------

def _call_api_with_retry(client, **kwargs):
    """レート制限や過負荷のエラーに備えて、APIコールを指数バックオフでラップします。"""
    delays = [1, 2, 4]
    last_err = None
    for attempt in range(len(delays) + 1):
        try:
            return client.messages.create(**kwargs)
        except anthropic.RateLimitError as e:
            last_err = e
        except anthropic.APIStatusError as e:
            if e.status_code == 529:
                last_err = e
            else:
                raise
        if attempt < len(delays):
            print(f" (retry {attempt+1} in {delays[attempt]}s)", end="", flush=True)
            time.sleep(delays[attempt])
    raise last_err


def run_single_prompt(client, system_prompt, user_input):
    result = _call_api_with_retry(
        client, model=MODEL, max_tokens=MAX_TOKENS, system=system_prompt,
        messages=[{"role": "user", "content": user_input}],
    )
    return result.content[0].text


def run_chain(client, prompts, user_input):
    output = ""
    step_outputs = []
    for i, system_prompt in enumerate(prompts):
        if i == 0:
            user_message = user_input
        else:
            parts = [f"ORIGINAL TICKET:\n{user_input}"]
            for j, prev_output in enumerate(step_outputs):
                parts.append(f"STEP {j+1} OUTPUT:\n{prev_output}")
            user_message = "\n\n".join(parts)
        try:
            output = run_single_prompt(client, system_prompt, user_message)
        except Exception as e:
            output = json.dumps({"error": f"Chain step {i+1} failed: {e}"})
        step_outputs.append(output)
    return output


def run_eval(client, prompts, cases_data, verbose=False):
    cases = cases_data["cases"]
    categories = cases_data["categories"]
    total = len(cases)
    results = []
    print(f"Running {total} eval cases...")
    for i, case in enumerate(cases):
        case_id = case["id"]
        print(f"  Case {case_id:>2}/{total}...", end="", flush=True)
        try:
            if len(prompts) == 1:
                raw_output = run_single_prompt(client, prompts[0], case["input"])
            else:
                raw_output = run_chain(client, prompts, case["input"])
        except Exception as e:
            raw_output = f"API ERROR: {e}"
        parsed, parse_error = parse_output(raw_output)
        response_coherent = None
        if case.get("audited_response") and parsed:
            priority = parsed.get("priority", "")
            response_text = parsed.get("response", "")
            if response_text:
                judge_pass, judge_reason = judge_response(client, case["input"], priority, response_text)
                response_coherent = judge_pass
        result = score_case(
            parsed=parsed, parse_error=parse_error, input_text=case["input"],
            gold_priority=case["gold_priority"], gold_entities=case["gold_entities"],
            response_coherent=response_coherent, audited=case.get("audited_response", False),
        )
        result["case_id"] = case_id
        result["category"] = case["category"]
        result["raw_output"] = raw_output
        results.append(result)
        status = "PASS" if result["pass"] else "FAIL"
        if result["pass"]:
            print(f" {status}")
        else:
            failed_criteria = [
                name for name, crit in result["criteria"].items()
                if not crit["pass"]
            ]
            reasons = [
                f'{name}: {result["criteria"][name]["reason"]}'
                for name in failed_criteria
            ]
            reason_str = "; ".join(reasons)
            print(f" {status} [{reason_str}]")
    category_results = {}
    for cat_key, cat_info in categories.items():
        cat_case_ids = cat_info["case_ids"]
        cat_results = [r for r in results if r["case_id"] in cat_case_ids]
        passed = sum(1 for r in cat_results if r["pass"])
        category_results[cat_key] = {
            "label": cat_info["label"],
            "passed": passed,
            "total": len(cat_results),
        }
    total_passed = sum(1 for r in results if r["pass"])
    return {
        "results": results, "categories": category_results,
        "total_passed": total_passed, "total_cases": total,
    }

# ---------------------------------------------------------------------------
# 可視化（visualization）
# ---------------------------------------------------------------------------

def _score_color(pct):
    if pct >= 80: return "#22c55e"
    if pct >= 60: return "#eab308"
    return "#ef4444"


def _display_results(eval_result, label="Current"):
    global _baseline_score, _baseline_total
    total_passed = eval_result["total_passed"]
    total_cases = eval_result["total_cases"]
    pct = round(100 * total_passed / total_cases) if total_cases else 0
    color = _score_color(pct)

    baseline_html = ""
    if _baseline_score is not None:
        bl_pct = round(100 * _baseline_score / _baseline_total)
        diff = total_passed - _baseline_score
        diff_str = f"+{diff}" if diff >= 0 else str(diff)
        diff_color = "#22c55e" if diff > 0 else ("#ef4444" if diff < 0 else "#6b7280")
        baseline_html = f'<div style="font-size:16px;color:#6b7280;margin-top:4px;">Baseline: {_baseline_score}/{_baseline_total} ({bl_pct}%) &rarr; Current: {total_passed}/{total_cases} ({pct}%) <span style="color:{diff_color};font-weight:bold;">[{diff_str}]</span></div>'

    html = f'''
    <div style="background:#f8fafc;border:2px solid {color};border-radius:12px;padding:24px;margin:16px 0;font-family:system-ui,-apple-system,sans-serif;">
      <div style="text-align:center;">
        <div style="font-size:14px;color:#6b7280;text-transform:uppercase;letter-spacing:1px;">{label}</div>
        <div style="font-size:64px;font-weight:bold;color:{color};">{total_passed}/{total_cases}</div>
        <div style="font-size:24px;color:{color};">({pct}%)</div>
        {baseline_html}
      </div>
    </div>
    '''
    _show_html(html)
    if not _in_notebook():
        line = f"[score] {label}: {total_passed}/{total_cases} ({pct}%)"
        if _baseline_score is not None:
            line += f"  |  baseline: {_baseline_score}/{_baseline_total}"
        print(line)
        return  # チャートはノートブック専用（notebook-only）

    cats = eval_result["categories"]
    labels = [cats[k]["label"] for k in cats]
    passed = [cats[k]["passed"] for k in cats]
    totals = [cats[k]["total"] for k in cats]
    failed = [t - p for t, p in zip(totals, passed)]

    plt = _viz()
    if plt is None:
        return
    fig, ax = plt.subplots(figsize=(10, 3.5))
    y_pos = range(len(labels))
    ax.barh(y_pos, passed, color='#22c55e', label='Passed', height=0.6)
    ax.barh(y_pos, failed, left=passed, color='#fee2e2', label='Failed', height=0.6)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=12)
    ax.set_xlabel('Cases', fontsize=12)
    ax.set_title('Score by Category', fontsize=14, fontweight='bold')
    ax.legend(loc='lower right')
    for i, (p, t) in enumerate(zip(passed, totals)):
        ax.text(t + 0.1, i, f'{p}/{t}', va='center', fontsize=11, fontweight='bold')
    ax.set_xlim(0, max(totals) + 1.5)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()


def _log_run(label, eval_result):
    _iteration_log.append({
        "timestamp": datetime.now().strftime("%H:%M:%S"),
        "label": label,
        "passed": eval_result["total_passed"],
        "total": eval_result["total_cases"],
        "pct": round(100 * eval_result["total_passed"] / eval_result["total_cases"]),
        "categories": {k: v.copy() for k, v in eval_result["categories"].items()},
    })

# ---------------------------------------------------------------------------
# 公開API：チームが使う関数はこれです
# ---------------------------------------------------------------------------

def run_eval_suite(prompt_or_prompts, label="Current"):
    """評価スイートをすべて実行し、結果を視覚的に表示します。

    引数：
        prompt_or_prompts: 単一のシステムプロンプト文字列、
                          またはチェーンモード用のプロンプト文字列のリスト。
        label: この実行のラベル（結果と反復ログに表示されます）。
    """
    global _baseline_score, _baseline_total

    client = anthropic.Anthropic()
    cases_data = _load_cases(_EVAL_CASES_B64)

    if isinstance(prompt_or_prompts, str):
        prompts = [prompt_or_prompts]
        mode = "single prompt"
    else:
        prompts = list(prompt_or_prompts)
        mode = f"{len(prompts)}-step chain"

    print(f"Mode: {mode}")
    print(f"Model: {MODEL}")
    print()

    start_time = time.time()
    eval_result = run_eval(client, prompts, cases_data)
    elapsed = time.time() - start_time

    print(f"\nCompleted in {elapsed:.1f}s")

    if _baseline_score is None:
        _baseline_score = eval_result["total_passed"]
        _baseline_total = eval_result["total_cases"]
        print(f"Baseline set: {_baseline_score}/{_baseline_total}")

    _log_run(label, eval_result)
    _display_results(eval_result, label=label)
    return eval_result


def compare_prompts(prompt_a, prompt_b, label_a="Variant A", label_b="Variant B"):
    """2つのプロンプトのバリエーションを実行し、並べて比較表示します。"""
    client = anthropic.Anthropic()
    cases_data = _load_cases(_EVAL_CASES_B64)

    prompts_a = [prompt_a] if isinstance(prompt_a, str) else list(prompt_a)
    prompts_b = [prompt_b] if isinstance(prompt_b, str) else list(prompt_b)

    print(f"Running {label_a}...")
    result_a = run_eval(client, prompts_a, cases_data)
    print(f"\nRunning {label_b}...")
    result_b = run_eval(client, prompts_b, cases_data)

    _log_run(label_a, result_a)
    _log_run(label_b, result_b)

    pct_a = round(100 * result_a["total_passed"] / result_a["total_cases"])
    pct_b = round(100 * result_b["total_passed"] / result_b["total_cases"])

    html = f'''
    <div style="display:flex;gap:16px;margin:16px 0;font-family:system-ui,-apple-system,sans-serif;">
      <div style="flex:1;background:#f8fafc;border:2px solid {_score_color(pct_a)};border-radius:12px;padding:20px;text-align:center;">
        <div style="font-size:13px;color:#6b7280;text-transform:uppercase;">{label_a}</div>
        <div style="font-size:48px;font-weight:bold;color:{_score_color(pct_a)};">{result_a["total_passed"]}/{result_a["total_cases"]}</div>
        <div style="font-size:20px;color:{_score_color(pct_a)};">({pct_a}%)</div>
      </div>
      <div style="flex:1;background:#f8fafc;border:2px solid {_score_color(pct_b)};border-radius:12px;padding:20px;text-align:center;">
        <div style="font-size:13px;color:#6b7280;text-transform:uppercase;">{label_b}</div>
        <div style="font-size:48px;font-weight:bold;color:{_score_color(pct_b)};">{result_b["total_passed"]}/{result_b["total_cases"]}</div>
        <div style="font-size:20px;color:{_score_color(pct_b)};">({pct_b}%)</div>
      </div>
    </div>
    '''
    _show_html(html)
    if not _in_notebook():
        print(f"[compare] {label_a}: {result_a['total_passed']}/{result_a['total_cases']} ({pct_a}%)  vs  {label_b}: {result_b['total_passed']}/{result_b['total_cases']} ({pct_b}%)")

    cats_a = result_a["categories"]
    cats_b = result_b["categories"]
    cat_keys = list(cats_a.keys())
    labels = [cats_a[k]["label"] for k in cat_keys]
    scores_a = [cats_a[k]["passed"] for k in cat_keys]
    scores_b = [cats_b[k]["passed"] for k in cat_keys]
    totals = [cats_a[k]["total"] for k in cat_keys]

    plt = _viz() if _in_notebook() else None
    if plt is not None:
        fig, ax = plt.subplots(figsize=(10, 4))
        y = range(len(labels))
        bar_height = 0.35
        ax.barh([i - bar_height/2 for i in y], scores_a, bar_height, label=label_a, color='#3b82f6')
        ax.barh([i + bar_height/2 for i in y], scores_b, bar_height, label=label_b, color='#f97316')
        ax.set_yticks(list(y))
        ax.set_yticklabels(labels, fontsize=12)
        ax.set_xlabel('Cases Passed', fontsize=12)
        ax.set_title('Category Comparison', fontsize=14, fontweight='bold')
        ax.legend(loc='lower right')
        for i, (sa, sb, t) in enumerate(zip(scores_a, scores_b, totals)):
            ax.text(max(sa, sb) + 0.15, i, f'/{t}', va='center', fontsize=10, color='#6b7280')
        ax.set_xlim(0, max(totals) + 1.5)
        ax.invert_yaxis()
        plt.tight_layout()
        plt.show()

    rows = ""
    for k in cat_keys:
        sa = cats_a[k]["passed"]
        sb = cats_b[k]["passed"]
        t = cats_a[k]["total"]
        diff = sb - sa
        winner = label_b if diff > 0 else (label_a if diff < 0 else "Tie")
        diff_color = "#22c55e" if diff > 0 else ("#ef4444" if diff < 0 else "#6b7280")
        diff_str = f"+{diff}" if diff > 0 else str(diff)
        rows += f'<tr><td>{cats_a[k]["label"]}</td><td>{sa}/{t}</td><td>{sb}/{t}</td><td style="color:{diff_color};font-weight:bold;">{diff_str}</td><td>{winner}</td></tr>'

    table_html = f'''
    <table style="border-collapse:collapse;width:100%;margin:16px 0;font-family:system-ui,-apple-system,sans-serif;">
    <thead><tr style="background:#f1f5f9;"><th style="padding:8px;text-align:left;">Category</th><th>{label_a}</th><th>{label_b}</th><th>Delta</th><th>Winner</th></tr></thead>
    <tbody>{rows}</tbody>
    </table>
    '''
    _show_html(table_html)
    if not _in_notebook():
        for k in cat_keys:
            d = cats_b[k]["passed"] - cats_a[k]["passed"]
            sign = "+" if d > 0 else ""
            print(f"  {cats_a[k]['label']}: {cats_a[k]['passed']} vs {cats_b[k]['passed']} of {cats_a[k]['total']}  ({sign}{d})")
    return result_a, result_b


def show_iteration_log():
    """反復の履歴を、表と折れ線グラフで表示します。"""
    if not _iteration_log:
        print("まだ反復が記録されていません。先に run_eval_suite() を実行してください。")
        return

    rows = ""
    for i, entry in enumerate(_iteration_log):
        color = _score_color(entry["pct"])
        rows += f'<tr><td>{i+1}</td><td>{entry["timestamp"]}</td><td>{entry["label"]}</td><td style="color:{color};font-weight:bold;">{entry["passed"]}/{entry["total"]} ({entry["pct"]}%)</td></tr>'

    table_html = f'''
    <h3 style="font-family:system-ui,-apple-system,sans-serif;">Iteration Log</h3>
    <table style="border-collapse:collapse;width:100%;margin:8px 0;font-family:system-ui,-apple-system,sans-serif;">
    <thead><tr style="background:#f1f5f9;"><th style="padding:8px;text-align:left;">#</th><th>Time</th><th>Label</th><th>Score</th></tr></thead>
    <tbody>{rows}</tbody>
    </table>
    '''
    _show_html(table_html)
    if not _in_notebook():
        for i, entry in enumerate(_iteration_log):
            print(f"  {i+1}. [{entry['timestamp']}] {entry['label']}: {entry['passed']}/{entry['total']} ({entry['pct']}%)")

    plt = _viz() if _in_notebook() else None
    if plt is not None and len(_iteration_log) > 1:
        fig, ax = plt.subplots(figsize=(10, 4))
        x = range(1, len(_iteration_log) + 1)
        pcts = [e["pct"] for e in _iteration_log]
        ax.plot(x, pcts, 'o-', color='#3b82f6', linewidth=2, markersize=8)
        ax.axhline(y=75, color='#22c55e', linestyle='--', alpha=0.7, label='Pass threshold (75%)')
        ax.axhline(y=90, color='#eab308', linestyle='--', alpha=0.7, label='Optimizer tier (90%)')
        for i, (xi, pct) in enumerate(zip(x, pcts)):
            ax.annotate(f'{pct}%', (xi, pct), textcoords="offset points",
                        xytext=(0, 12), ha='center', fontsize=10, fontweight='bold')
        ax.set_xlabel('Iteration', fontsize=12)
        ax.set_ylabel('Pass Rate (%)', fontsize=12)
        ax.set_title('Score Progression', fontsize=14, fontweight='bold')
        ax.set_ylim(0, 105)
        ax.set_xticks(list(x))
        ax.set_xticklabels([e["label"][:20] for e in _iteration_log], rotation=30, ha='right', fontsize=9)
        ax.legend(loc='lower right')
        plt.tight_layout()
        plt.show()


print("評価ハーネスを読み込みました。")
print()
print("使える関数：")
print("  run_eval_suite(prompt_or_prompts, label='...')  -- 評価を全件実行し、結果を視覚的に表示")
print("  compare_prompts(prompt_a, prompt_b, ...)        -- 2つのバリエーションをA/B比較")
print("  show_iteration_log()                            -- すべての実行とスコア推移のチャートを表示")


In [ ]:
# ============================================================
# 最初の評価を実行（RUN YOUR FIRST EVAL）：ベースラインのスコアを確立する
# ============================================================

results = run_eval_suite(system_prompt, label="Baseline (Broken Prompt)")

---

### 根本原因を名指しする前に：Claudeで診断を実行する

この場面では、Claudeは診断ツールとして役立ちます。何が悪いのかを決める前に、新しいClaudeの会話を開き、失敗例5件（上の「失敗の例」セクション）を次のプロンプトと一緒に貼り付けてください。

> 「サポートチケットの分類器が失敗している5つのケースです。どんな構造的なパターンが見えますか？ 最も可能性の高い根本原因は何でしょうか？」

出力は、プロファイラーの結果と同じように扱ってください。評価すべき仮説の集まりであり、そのまま受け入れる結論ではありません。挙げられた内容に同意できますか？ 見落としはありませんか？

そのうえで、先に進む前に、自分自身の診断を言葉にしてください。

---

## スコアの目安

| 段階 | 想定スコア | 意味 |
|-------|---------------|---------------|
| ベースライン（壊れたプロンプト） | 約4/21（19%） | 出発点です。改善の余地がたくさんあります |
| 良い単一プロンプトの修正 | 9〜12/21（43〜57%） | 主要な問題に対処できています |
| 優れたチェーンによる解決 | 16〜19/21（76〜90%） | 優秀です。手法を使いこなせています |
| 参照解答 | 21/21（100%） | 理想とする水準です |

ベースラインが低くても心配はいりません。そう設計されています。目標は、反復して改善することです。

## 診断のガイド

修正を始める前に、何が悪いのかを*理解する*ことに、しっかり時間をかけてください。実際の案件と同じように扱います。処方する前に診断します。

### 注目する場所

ベースライン評価の**カテゴリ別の内訳**を見てください。
- スコアが0のカテゴリはどれですか？ そこが、最も効果の大きい対象です。
- すでに合格しているカテゴリはどれですか？ それらは守ってください。難しいケースへの修正が、簡単なケースを壊すことはよくあります。

### 失敗例で探すパターン

1. **タスクの干渉**：このプロンプトは、分類、抽出、そして回答の作成を同時にモデルに求めています。回答を作成するタスクが共感的な表現を呼び起こすと、優先度の分類を押し上げてしまうことがあります。抽出で何も見つからないと、モデルはフィールドを埋めるためにエンティティを作り出します。

2. **エンティティのハルシネーション**：プロンプトには「空であっても、すべてのJSONフィールドを必ず含める」とあります。モデルは「含める」を「null のままにする」ではなく「埋める」と解釈します。曖昧なチケットは、でっち上げの製品名やエラーコードを誘発します。

3. **感情的な口調による優先度の偏り**：プロンプトはP1を「システム停止、全ユーザーに影響」と定義していますが、*口調を無視して内容に集中する*ようには指示していません。怒りや緊急を訴える言葉があると、実際の問題が機能リクエストであっても、優先度が押し上げられます。

4. **機能リクエストの誤分類**：緊急そうに聞こえる機能リクエストでもP4であることを示す明確な例がないため、モデルは緊急を示す言葉を高い優先度に結びつけてしまいます。

### メタプロンプティングを試す

壊れたプロンプトを、失敗例2〜3件と一緒に新しいClaudeの会話に貼り付けてください。そして、次のように尋ねます。

> 「このプロンプトのどんな構造上の問題が、これらの出力を引き起こしている可能性がありますか？ どの指示が曖昧で、矛盾していて、あるいは欠けているのかを、具体的に教えてください。」

これは、手持ちの道具の中でも特に強力な診断手法の1つです。Claudeを使って、そのClaude自身に向けたプロンプトを分析させましょう。

---

### どの手法が合うか分からない場合は、Claudeに聞く

先ほど診断した失敗パターンを取り上げて、Claudeに尋ねてください。

> 「[失敗の様子を書く]で失敗するプロンプトを直しています。メタプロンプティング、プロンプトチェーン、few-shot例、制約の明確化のうち、どれをおすすめしますか？ この特定の失敗パターンについて、理由も教えてください。」

そのうえで、最終的な判断は自分で下してください。大事なのはClaudeが正しいかどうかではありません。Claudeの推論が、あなたの診断に照らして成り立つかどうかです。

---

## ツールキット

4つの手法があり、それぞれ向いている失敗のタイプが異なります。診断に基づいて選んでください。ルーブリックが評価するのは手法の数ではなく、*根拠*です。

### 1. メタプロンプティング
**内容：**Claudeを使って、プロンプト自体を分析し、改善します。

**使う場面：**プロンプトが*なぜ*失敗するのか分からないとき、または曖昧さを体系的に洗い出したいときです。

**方法：**壊れたプロンプトと失敗例をClaudeに貼り付け、構造上の問題の特定、書き直しの提案、改善した指示の生成を依頼します。

---

### 2. プロンプトチェーン
**内容：**1つの巨大なプロンプトを、各ステップが1つの責務に絞られた複数ステップのパイプラインに分解します。

**使う場面：**プロンプトが一度に多くのことをこなそうとして、タスク同士が干渉しているときです（回答を下書きしながら分類すると、分類の精度が落ちる、など）。

**方法：**「優先度を分類 → エンティティを抽出 → 回答を下書き」のようなステップに分けます。各ステップには専用のシステムプロンプトを持たせます。評価ハーネスはチェーンモードに対応しているので、プロンプトのリストを渡してください。

**トレードオフ：**チェーンの各ステップは、それぞれ独立したAPI呼び出しです。お客様のレイテンシ上限は5秒なので、逐次の3回の呼び出しは、それぞれ約1.5秒で完了する必要があります。

---

### 3. A/Bテスト
**内容：**競合する複数のプロンプトのバリエーションを、同じ評価セットに対して実行し、定量的に比較します。

**使う場面：**どの言い回し、指示のセット、アーキテクチャがより良く機能するか分からないときです。

**方法：**`compare_prompts(prompt_a, prompt_b)`を使って両方のバリエーションを実行し、カテゴリ別の内訳を並べて確認します。

---

### 4. 自動プロンプト最適化
**内容：**体系的な反復ループです。バリエーションを生成し、それぞれを評価し、最良のものを選びます。

**使う場面：**改善が頭打ちになり、解の空間を体系的に探索する必要があるときです。

**方法：**（メタプロンプティングを使って）プロンプトのバリエーションを生成し、それぞれを評価し、最も成績の良いものを残すループを書きます。手作業版なら、最も難しい指示について3つのバリエーションを試し、3つとも評価して、勝ったものを残します。

In [ ]:
# ============================================================
# チェーンモード（CHAIN MODE）：複数ステップに分割した場合はこちらを使います
# ============================================================

# ステップ1：分類（Classifier）
step1_prompt = """
[ここに分類用のプロンプトを書いてください：優先度の分類だけに絞る]
"""

# ステップ2：エンティティ抽出（Entity Extractor）
step2_prompt = """
[ここにエンティティ抽出用のプロンプトを書いてください：製品名、バージョン、エラーコード、影響を受けるユーザー数を抽出する]
"""

# ステップ3：回答の下書き作成（Response Drafter）
step3_prompt = """
[ここに回答作成用のプロンプトを書いてください：お客様向けの回答を下書きし、最終的なJSONを組み立てる]
"""

# チェーンを実行する（準備ができたらコメントを外してください。uncomment）
# results = run_eval_suite([step1_prompt, step2_prompt, step3_prompt], label="Chain v1")

In [ ]:
# ============================================================
# A/B比較（A/B COMPARISON）：2つのプロンプトのバリエーションを比較します
# ============================================================

prompt_a = system_prompt  # 現時点でのベスト（best）

prompt_b = """
[ここにバリエーションを貼り付けてください]
"""

# 比較を実行する（準備ができたらコメントを外してください。uncomment）
# result_a, result_b = compare_prompts(prompt_a, prompt_b, label_a="Current Best", label_b="New Variant")

In [ ]:
# ============================================================
# 反復ログ（ITERATION LOG）：実行ごとの進捗を追跡します
# ============================================================

# 評価ハーネス（eval harness）は、各実行を自動的に記録します。
# show_iteration_log() を呼び出すと、反復の履歴とスコアの推移を確認できます：

show_iteration_log()

---

### 行き詰まったら：人に聞く前に、Claudeに聞く

何を試して、何が起きたのかを、Claudeに正確に伝えてください。

> 「プロンプトを反復改善しています。[変更内容を書く]を試したところ、スコアがXからYになりました。まだ失敗しているカテゴリは[一覧]です。次に何を試しますか？ その理由も教えてください。」

答えを求めているのではありません。診断を実行しているのです。提案に従う前に、自分の診断に照らして評価してください。

---

## 振り返り：あなたが担った部分とClaudeが担った部分

ここが本当の学びの場であり、優れたパートナーエンジニアが差をつける場所です。

スコアを共有する前に、数分かけて、次の質問に正直に答えてください。

### 1. プロンプトエンジニアリングが効いたのはどこですか？
どの改善が、あなた自身の意図的な構造上の判断から生まれましたか？ チェーンのステップ、明示的な制約、few-shot例、nullの扱いの指示などです。

具体例を2〜3個書いてください。*「Yという失敗モードを診断したので、Xを追加した」*という形です。

### 2. Claudeの基礎能力が効いたのはどこですか？
改善の中には、Claudeがもともと得意なことが効いたものもあります。あなたはその邪魔をしなくなっただけです。この違いについて、正直になってください。*「すべてのフィールドを埋めさせるのをやめたら、Claudeは英語が母語でない人の文章もうまく扱えた」*といった具合です。

### 3. 何に驚きましたか？
期待どおりに動かなかったものは何ですか？ Claudeが予想より良かった点、あるいは悪かった点は何ですか？

### 4. CTOには何と伝えますか？
技術的な意思決定者に、自分のアーキテクチャを90秒で説明するとしたら、何と言いますか？ 共有する前に、ここに下書きしてください。

---

## Claudeから個別の分析を受ける

振り返りを終えたら、最終スコアと回答を、次のプロンプトを使って新しいClaudeの会話に貼り付けてください。

```
I just completed a prompt engineering exercise. Here's my summary:

Final score: [X/21 — Y%]
Baseline: [X/21 — Y%]

Techniques I used:
[List them + brief rationale for each]

Where my engineering did the work:
[Your answer from question 1]

Where Claude's capability did the work:
[Your answer from question 2]

What surprised me:
[Your answer from question 3]

Please give me a candid analysis:
1. Where did my prompt engineering approach shine?
2. Where did I rely on Claude's capability when I should have engineered more explicitly?
3. What would make my approach more defensible in an enterprise context?
4. What's one technique I didn't use that would likely push my score higher?
```

---

## 発展課題：プロセス監査

*高いスコアに到達した、早く終わった人向けです。任意です。*

これは、方法論とデモを分ける問いです。**作業をしたのはあなたですか、それともClaudeですか？**

### ステップ1：Claudeにプロセスを監査してもらう

新しいClaudeの会話を開き、次を送信してください。

```
Here's my original prompt:
[paste the broken prompt from the ✏️ YOUR TURN cell]

Here's my final prompt (or chain):
[paste your best version]

Score progression: [paste from your iteration log]

How much of this improvement came from diagnostic decisions I made — and how much would have happened if you'd just rewritten the prompt from scratch without the eval loop?
```

### ステップ2：実験を実行する

元のプロンプトをClaudeに直接直させます。方法論は使わず、ただこう伝えてください。*「これは壊れたサポートチケット分類器です。失敗を直すように書き直してください。改善したプロンプトだけを返してください。」*

Claudeの書き直しを下のセルに貼り付けて、実行してください。

### ステップ3：比較する

スコアは同じですか？ 違いますか？ どちらでも、本当の問いは同じです。来週、別の壊れたプロンプトに出会ったときにも繰り返せる形で、自分のやり方が*なぜ*うまくいくのかを説明できますか？

説明できるスコアは、方法論です。説明できないスコアは、デモです。

---

In [ ]:
# ============================================================
# プロセス監査：あなたのやり方とClaudeの直接修正を比較します
# ============================================================

claude_fix = """
[ここにClaudeが直接書き直したものを貼り付けてください]
"""

results_audit = run_eval_suite(claude_fix, label="Claude direct fix")

## エンタープライズのお客様への伝え方

この演習の目的は、スコアを上げることだけではありません。技術に詳しいお客様に、実際のプロンプトエンジニアリングの案件を最初から最後まで説明できる自信をつけることです。その方法は次のとおりです。

### 語りの流れ

| 場面 | 伝える内容 | うまくいく理由 |
|------|-------------|--------------|
| **問題** | 「御社のプロンプトは、一度に多くのことをこなそうとしていて、タスク同士が干渉しています。」 | 症状ではなく、根本原因を診断しています。 |
| **診断** | 「Claudeを使って、Claude自身の失敗モードを分析しました。特定された内容はこちらです。」 | メタプロンプティングの実践を示せます。お客様の反応が良い部分です。 |
| **修正** | 「[N]ステップのチェーンに分解しました。まず分類し、次に抽出し、最後に回答します。各ステップは役割が絞られていて、互いに影響を与えません。」 | プロンプトの微調整ではなく、アーキテクチャの思考を示せます。 |
| **実証** | 「改善前の合格率は19%、改善後は86%です。カテゴリ別の内訳はこちらです。機能リクエストが最も改善しました。口調ではなく内容を評価するようになったためです。」 | 定量的で、具体的で、説明に耐えます。 |
| **再発防止** | 「新しいプロンプトを本番に出す前に、チームが毎回実行できる、3つの質問のチェックリストです。」 | お客様が実際に使えるものを残せます。 |

### 良い説明と優れた説明の違い

- **良い説明：**「プロンプトを直したら、スコアが上がりました。」
- **優れた説明：**「構造的な失敗モードを診断し、それに的確に対応する手法を選び、失敗カテゴリごとの差分をお見せできます。」

お客様との契約を決めるのは、後者の説明です。

### 決して言ってはいけないこと

「Claudeに直させただけです」とは、決して言わないでください。メタプロンプティングが役に立ったとしても、診断の判断は自分のものとして語ってください。何を尋ねるかを決めたのはあなたです。結果を評価したのもあなたです。どの修正をリリースするかを決めたのもあなたです。それが、あなたがもたらす価値です。